## Prepare Data

In [ ]:
import os
import numpy as np
import tiktoken
import json
import multiprocessing as mp
from collections import deque
from itertools import islice

from datasets import load_dataset
from tqdm import tqdm

# Config parameters
DATASET_NAME = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample-10BT"
OUTPUT_DIR = "edu_fineweb"
SHARD_SIZE = 10_000_000          # 20 MB per shard (uint16)
MAX_TOKENS = 1_000_000_000       # stop after this many tokens (None = no limit)
NUM_WORKERS = max(1, (os.cpu_count() or 2) // 2)   # leave half the CPU free
BATCH_DOCS = 64                  # documents per task sent to a worker
MAX_IN_FLIGHT = NUM_WORKERS * 4  # bounds RAM: batches queued at once
STATE_PATH = os.path.join(OUTPUT_DIR, "state.json")
CARRY_PATH = os.path.join(OUTPUT_DIR, "carry.npy")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initializee BPE encoder. Each worker process runs this once at startup.
def _init_worker():
    global _enc, _eot
    _enc = tiktoken.get_encoding("gpt2")
    _eot = _enc._special_tokens["<|endoftext|>"]

def tokenize_batch(texts):
    """Encode a batch of docs; each is prefixed with <|endoftext|>.
    Returns one flat uint16 array (cheap to send between processes)."""
    out = []
    for ids in _enc.encode_ordinary_batch(texts, num_threads=1):
        out.append(_eot)
        out.extend(ids)
    return np.array(out, dtype=np.uint16)

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            s = json.load(f)
        carry = np.load(CARRY_PATH) if os.path.exists(CARRY_PATH) else np.empty(0, np.uint16)
        return s, carry
    return {"shard_idx": 0, "docs_done": 0, "total_tokens": 0}, np.empty(0, np.uint16)

def save_state(state, carry):
    np.save(CARRY_PATH, carry)
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f)
    os.replace(tmp, STATE_PATH)

def batches(dataset):
    it = (d["text"] for d in dataset)
    while True:
        chunk = list(islice(it, BATCH_DOCS))
        if not chunk:
            return
        yield chunk   

In [ ]:
print(f"Streaming dataset: {DATASET_NAME} ({DATASET_SUBSET})...")
# Stream data to prevent storing raw uncompressed text on local disk
dataset = load_dataset(DATASET_NAME, name=DATASET_SUBSET, split="train", streaming=True)

shard_idx = 0
token_buffer = []

for doc in tqdm(dataset, desc="Tokenizing documents"):
    tokens = tokenize_document(doc)
    token_buffer.extend(tokens)

Streaming dataset: HuggingFaceFW/fineweb-edu (sample-10BT)...


Tokenizing documents: 304000it [04:47, 1055.69it/s]
